# Explore the ApprenticeOps 152-model run

Companion notebook for run **`full-chatok-core20-r5-ollama-20260705-150053`** — 152 small local LLMs × 20 homelab-ops scenarios × 5 reps, scored by two independent LLM judges, with per-task energy on every row (single CPU regime, RAPL package-0).

**Two ways to use it:**

- **Path A — light (default).** Loads the compact, redaction-safe snapshots that ship in the repo (`data/snapshots/*.csv`, ~3.5 MB). No download. Enough to reproduce the headline tables and play with the data.
- **Path B — full raw bundle (optional).** Downloads the 434 MB tar from Azure Blob (or uses a local copy), verifies its sha256, extracts it, and loads every model's *raw* outputs + both judges' *raw* verdicts.

> The artifact is already-gzipped JSONL inside an uncompressed `.tar` (re-gzipping saves ~5%, so it's left plain). `pandas` reads the inner `.jsonl.gz` files directly — no manual unzip needed.

In [ ]:
import hashlib, tarfile, urllib.request
from pathlib import Path
import pandas as pd, numpy as np

# --- locate the repo root whether this runs from docs/analysis/ or the repo root ---
def find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "snapshots").is_dir():
            return p
    return start
REPO = find_repo(Path.cwd())

RUN_ID    = "full-chatok-core20-r5-ollama-20260705-150053"
BUNDLE_ID = "dd262a5c94593cb4b35bbb3554cc7ed1d608fab8b16160a3215329637c614baa"

# Light path (already in git, ~3.5 MB total) — no download needed:
RESULTS_CSV = REPO / "data" / "snapshots" / f"{RUN_ID}.results.csv"
JUDGED_CSV  = REPO / "data" / "snapshots" / f"{RUN_ID}.judged.csv"

# Full raw bundle (434 MB tar of already-gzipped JSONL) hosted on Azure Blob.
# Paste the download URL once exposure is chosen (anonymous-read URL or a SAS URL):
BLOB_URL = ""   # e.g. "https://ceopsdata644948.blob.core.windows.net/full-run-152/full-chatok-core20-r5-ollama-20260705-150053.bundle.tar[?<sas>]"
EXPECTED_TAR_SHA256 = "f509ab5de419dff16ec59fc8197eb805fc41bacd3d46b90aae3ce5da1e5040f0"
# If you already have the bundle locally, this is where the loader looks first:
LOCAL_BUNDLE = REPO / "data" / "completed-runs" / f"{RUN_ID}-{BUNDLE_ID}"

print("repo:", REPO)
print("light path present:", RESULTS_CSV.exists(), JUDGED_CSV.exists())
print("local full bundle present:", LOCAL_BUNDLE.exists())

## Path A — light: the snapshots in the repo

`results` = one row per (model, scenario, rep) with determinism score, energy, throughput. `judged` = the two judges' 0–5 scores per run. The **quality axis** is their consensus (mean of the two judges).

In [ ]:
# --- Path A: load the compact, redaction-safe snapshots (metrics + judge scores only) ---
results = pd.read_csv(RESULTS_CSV)   # 15,200 rows = 152 models x 20 scenarios x 5 reps
judged  = pd.read_csv(JUDGED_CSV)    # 30,400 rows = the two judges' scores per run
print("results:", results.shape, "| judged:", judged.shape)
print("models:", results.model.nunique(),
      "| scenarios:", results.scenario.nunique(),
      "| judges:", sorted(judged.judge_model.unique()))

# Quality axis = mean of the two judges (consensus) per run, then per model.
consensus = (judged.groupby(["model", "scenario", "rep"], as_index=False)
                    .score.mean().rename(columns={"score": "quality"}))

model_table = (consensus.groupby("model").quality
                        .agg(quality="mean", quality_sd="std", n_runs="count"))
agg = results.groupby("model").agg(det_score=("det_score", "mean"),
                                   energy_wh=("energy_wh", "mean"),
                                   decode_tps=("decode_tokens_per_s", "mean"))
model_table = model_table.join(agg)
model_table["quality_per_wh"] = model_table.quality / model_table.energy_wh
model_table = model_table.sort_values("quality", ascending=False)

print("\nTop 15 models by consensus quality (0-5):")
model_table.head(15).round(3)

In [ ]:
# --- Play: three quick questions the dataset can answer ---
import matplotlib.pyplot as plt

# 1) Which ops tasks are hardest? (lowest mean consensus quality)
scen = consensus.groupby("scenario").quality.mean().sort_values()
print("Hardest 5 scenarios:\n" + scen.head(5).round(2).to_string())
print("\nEasiest 5 scenarios:\n" + scen.tail(5).round(2).to_string())

# 2) Most energy-efficient *capable* models (quality per watt-hour, among quality >= 2.5)
eff = model_table[model_table.quality >= 2.5].sort_values("quality_per_wh", ascending=False)
print("\nTop 8 by quality-per-Wh (quality >= 2.5):\n" +
      eff[["quality", "energy_wh", "quality_per_wh"]].head(8).round(3).to_string())

# 3) The efficiency frontier: one dot per model, energy vs quality
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(model_table.energy_wh, model_table.quality, s=14, alpha=0.6)
ax.set_xscale("log")
ax.set_xlabel("energy per task (Wh, log scale)")
ax.set_ylabel("consensus quality (0-5)")
ax.set_title("152 small models: quality vs energy (single CPU regime)")
best = model_table.head(1)
ax.annotate(best.index[0], (best.energy_wh.iloc[0], best.quality.iloc[0]),
            fontsize=8, xytext=(5, -8), textcoords="offset points")
plt.tight_layout(); plt.show()

## Path B — full raw bundle (optional)

Set `BLOB_URL` in the config cell to the Azure link (anonymous-read URL or a SAS URL) — or drop the bundle locally — to load the complete raw dataset: **every generated answer and every judge verdict**. The cell below resolves a source (local copy → download+verify → skip), extracts the tar, and reads the inner `canonical/*.jsonl.gz` straight into pandas.

In [ ]:
# --- Path B: resolve a source -> (local bundle | download+verify+extract) -> load raw ---
def sha256_file(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

bundle_dir = None
if LOCAL_BUNDLE.exists():
    bundle_dir = LOCAL_BUNDLE
    print("using local bundle:", bundle_dir)
elif BLOB_URL:
    tar_path = Path("/tmp") / f"{RUN_ID}.bundle.tar"
    if not tar_path.exists():
        print("downloading 434 MB ..."); urllib.request.urlretrieve(BLOB_URL, tar_path)
    got = sha256_file(tar_path)
    assert got == EXPECTED_TAR_SHA256, f"sha256 mismatch!\n got {got}\n exp {EXPECTED_TAR_SHA256}"
    print("sha256 verified:", got)
    extract_to = Path("/tmp") / "ceops-152-bundle"; extract_to.mkdir(exist_ok=True)
    with tarfile.open(tar_path) as t:
        t.extractall(extract_to)
    bundle_dir = extract_to / f"{RUN_ID}-{BUNDLE_ID}"
else:
    print("No local bundle and BLOB_URL is empty -> set BLOB_URL to the Azure link to fetch the full raw data.")

if bundle_dir:
    # canonical files are gzipped JSON Lines -> pandas reads them directly (no manual unzip).
    raw_results = pd.read_json(bundle_dir / "canonical" / "results.jsonl.gz", lines=True)
    raw_judged  = pd.read_json(bundle_dir / "canonical" / "judged.jsonl.gz",  lines=True)
    print("raw results:", raw_results.shape, "| raw judged:", raw_judged.shape)
    print("a few raw result columns:", [c for c in raw_results.columns][:14])
    display(raw_results.head(3))